# 12 · SFT with more training data (`sft_100k`)

**The question.** Our SFT model (`sft_v1`) learned from 20,000 positions, 14% of the 142,500 training positions we have. It solves 7.8% of the test puzzles, only a little above a random legal move (5.6%), and 28% of its answers are illegal. This experiment asks: **how much better does SFT get with more data?**

This matters for every later method. DPO and RL do not teach the model new moves; they shift probability between moves the model already considers. When SFT gives the best move only a small chance (about 1.5% for Stockfish's move at the start of DPO, notebook 09), there is little for them to improve. A stronger SFT model is a better starting point for both.

**What changes, and what stays the same.** Only the amount of training data changes:

| | `sft_v1` (notebook 03) | `sft_100k` (this notebook) |
|---|---|---|
| Training positions | 20,000 | **100,000** (5 times as many; the first 20,000 are exactly `sft_v1`'s, in the same order) |
| Answer the model learns | the move the player made | the same |
| Model, LoRA, learning rate, epochs (1), seed, prompt | `configs/sft.yaml` | the same |
| Graded on | 1,000 test puzzles | the same 1,000 puzzles, same grader |

The only other difference is that the validation loss is measured every 500 steps instead of every 100, to save time; this does not affect training.

**One run gives the whole curve.** Instead of training separate models on 20k, 40k, 60k, 80k and 100k examples, we train once on 100,000 and save a copy of the model after 20k, 40k, 60k and 80k examples. Each copy is graded on the puzzles, which gives **puzzle accuracy against the number of training examples**. One caveat: the learning rate falls gradually over the whole run (a cosine schedule), so the copy after 20k examples has not finished that schedule, while `sft_v1` (trained on the same 20k) has. The copies are therefore close to, but not exactly, "a model trained on N examples". Comparing the 20k copy with `sft_v1` shows how large this effect is.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Plain *Serverless* has no GPU. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run. (The first cell restarts Python once; if Run all stops there, click Run all again.)
3. Set `QUICK_CHECK = False` and run the real thing as a **Job**, so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification. Separate Jobs can run at the same time.
4. Afterwards, commit and push **only `results/`** from the Git folder. Leave `notebooks/` unticked: Databricks cannot save notebook outputs to git, so committing a notebook from there would erase its outputs on GitHub. The outputs of a Job run are on the Job's run page; every figure and table is also saved in `results/figures/`.

**Time:** about 5 hours of training on an A10 and about 20 minutes of grading, roughly $20–25 of credits. (We have 142,500 training positions in total; 100,000 keeps the run to an afternoon.) If the Job stops early, the copies saved so far are kept in `checkpoints/` and can still be graded.

**Outputs**
- the adapter in `checkpoints/sft_100k/` and the copies in `checkpoints/sft_100k_at_20k/`, `_at_40k/`, `_at_60k/`, `_at_80k/` (not on GitHub)
- results: `results/sft_100k_training.json`, and for the final model and each copy `results/<name>.json`, `results/<name>_dev.json`, and rows in `results/summary.csv`
- every figure and table of this experiment in `results/figures/sft_100k/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
RUN_NAME = "sft_100k"
QUICK_CHECK = False   # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.
TRAIN_EXAMPLES = 100_000                # THE change in this experiment (sft_v1: 20,000)
SAVE_AT = [20_000, 40_000, 60_000, 80_000]  # save and grade a copy after this many examples

import gc

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import compare_with, figures_dir, grade_puzzles, puzzle_table, save_table, save_training
from chesslm.train_sft import copy_dir, examples_label, sample_examples, to_dataset, train

sft_cfg = load_config("sft")
prompt_cfg, eval_cfg = load_config("prompt"), load_config("eval")
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
SEED = sft_cfg["training"]["seed"]
SFT_V1_EXAMPLES = sft_cfg["data"]["train_examples"]          # 20,000
train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
sft_cfg["data"]["train_examples"] = TRAIN_EXAMPLES
sft_cfg["training"]["eval_steps"] = 500                      # validation loss less often, to save time

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUN_NAME += "_check"
    sft_cfg["data"].update(trial_examples=16, train_examples=96, val_examples=16)
    sft_cfg["training"].update(logging_steps=1, eval_steps=1)
    SAVE_AT = [32, 64]
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
ADAPTER_DIR = REPO_ROOT / "checkpoints" / RUN_NAME
FIGURES = figures_dir(RUN_NAME)
print(f"Run: {RUN_NAME} | {sft_cfg['data']['train_examples']:,} training examples | copies after {SAVE_AT} | "
      f"figures and tables -> {FIGURES}")

## 1. Training data: 100,000 positions, starting with `sft_v1`'s

A seeded random sample of 100,000 of the 142,500 training positions. With the same seed, the first 20,000 are exactly the positions `sft_v1` learned from, in the same order; the cell below checks this. So the copy saved after 20,000 examples has seen exactly `sft_v1`'s data. The 1,000 validation positions are the same as `sft_v1`'s.

In [ ]:
n = sft_cfg["data"]
train_rows = sample_examples(train_positions, n["train_examples"], SEED)
sft_v1_rows = sample_examples(train_positions, SFT_V1_EXAMPLES, SEED)
same_start = list(train_rows.fen[:SFT_V1_EXAMPLES]) == list(sft_v1_rows.fen) if not QUICK_CHECK else True
assert same_start, "the first 20,000 positions should be sft_v1's"
val_rows = sample_examples(pd.read_parquet(REPO_ROOT / "data" / "processed" / "val.parquet"), n["val_examples"], SEED)

data_table = pd.DataFrame({"Training positions": [SFT_V1_EXAMPLES, len(train_rows)],
                           "Share of all training positions (%)": [round(SFT_V1_EXAMPLES / len(train_positions) * 100, 1),
                                                                   round(len(train_rows) / len(train_positions) * 100, 1)]},
                          index=pd.Index(["sft_v1", RUN_NAME], name="Model"))
save_table(data_table, FIGURES, "training_data",
           "Training positions per model. The first 20,000 positions of sft_100k are exactly sft_v1's, in the same order.")
display(data_table)

## 2. Trial run

A short run on the first few hundred training examples, to check that the loss goes down on this machine before the long run.

In [ ]:
trial_ds = to_dataset(train_rows.head(n["trial_examples"]), prompt_cfg)
val_ds = to_dataset(val_rows, prompt_cfg)
trial_log, trial_compute = train(trial_ds, val_ds.select(range(min(100, len(val_ds)))), sft_cfg,
                                 REPO_ROOT / "checkpoints" / f"{RUN_NAME}_trial")
losses = trial_log.loss.dropna()
print(f"Training loss: {losses.iloc[0]:.3f} at the start -> {losses.iloc[-1]:.3f} at the end")
gc.collect()
torch.cuda.empty_cache()

## 3. Full training run, saving copies along the way

The figure shows the loss on training examples and on the 1,000 held-out validation positions. With five times more data, the validation loss should keep falling for longer than in `sft_v1`'s run; where it levels off is where more data stops helping.

In [ ]:
train_ds = to_dataset(train_rows, prompt_cfg)
print(f"train: {len(train_ds):,} examples  validation: {len(val_ds):,} examples")
log, compute = train(train_ds, val_ds, sft_cfg, ADAPTER_DIR, save_at=SAVE_AT)
save_training(RUN_NAME, {"sft": sft_cfg, "prompt": prompt_cfg, "save_at": SAVE_AT}, compute, log)
plots.loss_curve(log, FIGURES / "training_loss.png", title=f"{RUN_NAME}: loss during training");

## 4. Grade every copy and the final model on the puzzles

Exactly the grader, puzzles, and greedy decoding used for every other model: the 200 dev puzzles first, then the 1,000 test puzzles. Copies that are missing (for example, if training stopped early) are skipped.

In [ ]:
models = {f"{RUN_NAME}_at_{examples_label(k)}": (k, copy_dir(ADAPTER_DIR, k)) for k in SAVE_AT}
models[RUN_NAME] = (n["train_examples"], ADAPTER_DIR)
graded_models = {}
for name, (examples, adapter) in models.items():
    if not adapter.exists():
        print(f"{name}: skipped, {adapter} not found")
        continue
    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=adapter)
    grade_puzzles(model, tokenizer, "dev", f"{name}_dev", configs, str(adapter), PUZZLE_LIMIT)
    grade_puzzles(model, tokenizer, "test", name, configs, str(adapter), PUZZLE_LIMIT)
    graded_models[name] = examples
    del model

## 5. Results: does more data help?

**How to read this section**
- **The scaling curve** shows puzzle accuracy and the share of legal answers against the number of training examples seen. Dashed lines mark `sft_v1` and a random legal move (5.6%). A curve that is still rising at the right edge means even more data would likely help; a flat curve means data is no longer the limit.
- **The table** gives the same numbers, plus ACPL (how much worse the legal answers are than Stockfish's best move; lower is better).
- **The paired comparisons** count puzzles solved by only one of two models; a p-value below 0.05 means the difference is unlikely to be chance. `sft_100k` vs. `sft_v1` measures the effect of the extra data. The 20k copy vs. `sft_v1` (the same data, different learning-rate schedule) shows how much the schedule caveat matters.

All numbers are on the 1,000 test puzzles.

In [ ]:
copies = list(graded_models)
scaling = puzzle_table(copies).reset_index()
scaling.insert(1, "Training examples", [graded_models[name] for name in copies])
scaling = scaling.set_index("Training examples")[["Model", "Puzzle accuracy (%)", "Legal (%)", "ACPL (cp)"]]
save_table(scaling, FIGURES, "scaling",
           "Test-puzzle results of the copies saved during training and of the final model. ACPL is over legal answers only.")
display(scaling)

sft_v1 = puzzle_table(["sft_v1"]).iloc[0]
plots.scaling_curve(scaling, [("Puzzle accuracy (%)", "puzzle accuracy (%)"), ("Legal (%)", "legal answers (%)")],
                    {"sft_v1 (20k examples)": sft_v1.to_dict(), "random legal move": {"Puzzle accuracy (%)": 5.6}},
                    FIGURES / "scaling_curve.png", title="Does SFT improve with more training data?");

In [ ]:
runs = ["base", "sft_v1", RUN_NAME]
display(puzzle_table(runs, FIGURES))

pairs = [(RUN_NAME, "sft_v1")]
first_copy = f"{RUN_NAME}_at_{examples_label(SAVE_AT[0])}"
if first_copy in graded_models:
    pairs.append((first_copy, "sft_v1"))
comparison = pd.DataFrame([compare_with(new, old) for new, old in pairs]).set_index("model")
save_table(comparison, FIGURES, "paired_comparison",
           "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
display(comparison)

In [ ]:
summary = pd.read_csv(experiments.RESULTS / "summary.csv")
plots.compare_metrics(summary, runs, FIGURES / "puzzle_metrics.png",
                      title="20,000 (sft_v1) vs. 100,000 (sft_100k) training examples");
plots.accuracy_by_band(summary, runs, FIGURES / "accuracy_by_band.png",
                       title="Puzzle accuracy by puzzle difficulty (rating band)");
plots.outcome_breakdown(summary, runs, FIGURES / "answer_outcomes.png", title="Are the answers legal moves?");

## Summary of outputs

- `results/figures/sft_100k/`: `training_data` (table), `training_loss.png`, `scaling` (table) and `scaling_curve.png`, `puzzle_results` (table), `paired_comparison` (table), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`
- `results/sft_100k.json` and `results/sft_100k_at_<N>k.json`: every test answer and its grade; `results/sft_100k_training.json`: settings, compute, training log
- The final model is also graded on held-out game positions and in full games in notebook 11.
- If `sft_100k` beats `sft_v1`, it is the natural starting model for the combinations in notebook 15.